## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

import json

import glob
import os
from pathlib import Path

import pickle

## Importing data

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
with open('../raw_data/pickles/dc-4-eumaco.pickle', 'rb') as file:
	json_files_dict = pickle.load(file)

# List of all the json files names
json_filenames = list(json_files_dict)

Shows a sample of the field where the useful data is stored 

In [ ]:
json_files_dict[json_filenames[0]]['transmissions']['SpO2Transmission'][0][0]

## Analysis

Among three transmission fields...

In [ ]:
fields_to_check = json_files_dict[json_filenames[0]]['transmissions'].keys()
list(fields_to_check)

... choose between one of them (`SpO2Transmission`, `WeightTransmission` and `BloodPressureTransmission`)

In [ ]:
field_to_check = 'SpO2Transmission'

### Functions that returns the structure of a field in a json/dictionary

Variant where iterable are NOT analyzed recursively

In [ ]:
# def show_structure(data):
#     '''Takes in input a dictionary and returns a version of it where:
#     - keys are kept unchanged
#     - values are substituted with the type name (iterable are NOT analyzed recursively)'''

#     result = {}

#     for key, val in data.items():
#         if isinstance(val, dict):
#             result[key] = show_structure(val)  # Recursive call
#         else:
#             result[key] = type(val).__name__
#     return result

Variant where iterable are analyzed recursively

In [ ]:
from collections.abc import Mapping, Sequence

def show_structure(data):
    '''Takes in input a dictionary and returns a version of it where:
    - keys are kept unchanged
    - values are substituted with the type name (iterable are analyzed recursively)'''
    if isinstance(data, Mapping):
        result = {}
        for key, val in data.items():
            result[key] = show_structure(val)
        return result

    if isinstance(data, Sequence) and not isinstance(data, (str, bytes)):
        result = []
        for item in data:
            result.append(show_structure(item))
        return result

    return type(data).__name__


### Function that returns overview of the consistency of a field among json files

In [ ]:
def overview_of_field(json_files_dict, json_filenames, field_to_check, *, position=True):
    '''
    Takes in input:
    - a list of json filenames
    - a dictionary {json_filename: json_file_content}
    - the transmission subfield to check
    - the position option
    - a sample instance to check against

    Returns an overview dictionary {'different': {json_file: [position, ...]}, 'missing': [json_files]} that tells:
    - given a json file which have the field different from the sample one:
        - the instance's position within the file, if position=True (default behaviour)
        - the actual instance structure, if position=False
    - which json files misses the transmission subfield to check
    '''

    # The returned object will have this structure:
    # {'different': {json_file: [position, ...]}, 'missing': [json_files]}
    overview = {'different': {}, 'missing': []}

    # Defines a sample filed to use for the comparison
    sample = show_structure(json_files_dict[json_filenames[0]]['transmissions'][field_to_check][0][0])

    # Iterates over the jsons
    for json_name in json_filenames:

        # If the transmission field to check is NOT present in the json...
        # ... append the json name to the missing list
        if field_to_check not in json_files_dict[json_name]['transmissions'].keys():
            overview['missing'].append(json_name)
        
        # If the transmission field to check is present in the json...
        else:

            # Iterate over the long list in the json and set:
            # - t_position as the transmission field instance's position
            # - t_instance as the the transmission field itself
            for t_position, t_instance in enumerate(json_files_dict[json_name]['transmissions'][field_to_check][0]):

                # Get the sample of the field
                formatted_tr_inst = show_structure(t_instance)

                # If the sample is different from the current instance...
                if sample != formatted_tr_inst:
                    # ... add the instance's position to the {json_file: [position, ...]}}
                    if position == True:
                       overview['different'].setdefault(json_name, []).append(t_position)
                    else:
                        overview['different'].setdefault(json_name, []).append(formatted_tr_inst)
    return overview

### Run the overview among the three fields

In [ ]:
overview = overview_of_field(json_files_dict, json_filenames, field_to_check, position=False)

In [ ]:
overview

### Check consistency among the useful fields of the transmissions

Write the "show structure" json to file

In [ ]:
# data = digger_json(
#     json_files_list[0]
#     )

# with open("campion.json", "w") as f:
#     json.dump(data, f, indent=4)